[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/chreissel/sbi-tutorial-iaifi26/blob/main/03_hackathon_stellar_streams.ipynb)

# Notebook 3 — Milky-Way stellar streams (hackathon)

**IAIFI Summer School · simulation-based inference · full-day hackathon.**

Notebooks 1 and 2 built the machinery: a flow posterior, a learned data
embedding, and `falcon`'s "the model is a graph in YAML" workflow. This notebook
points all of it at a real astrophysics problem and then **hands you the keys** —
it is deliberately short. You get a working forward model and one worked warm-up;
the inference itself is yours to build.

A **stellar stream** is what is left when a star cluster or dwarf galaxy is
torn apart by the Milky Way's tides: its stars stretch into a thin ribbon
tracing the orbit. The shape, width, and kinematics of that ribbon encode the
progenitor (its mass and age) *and* the Galactic potential the stream fell
through — which is why streams are one of our best handles on the distribution
of **dark matter**. We use [`sstrax`](https://github.com/undark-lab/sstrax), a
`jax` simulator of the **GD1** stream, wrapped for `falcon` exactly the way
notebook 2 wrapped the gravitational-wave chirp.

> ### ▶️ Run this cell first
> **Runtime → Change runtime type → T4 GPU** helps the flow/CNN training, but
> the simulator itself is CPU-bound `jax`, so a CPU runtime works too. Like
> notebook 2 we **clone** the repo (falcon reads configs and writes samples to
> disk) and install the physics package `sstrax`.
>
> **If you forked or renamed this repository**, change `chreissel/sbi-tutorial-iaifi26`
> below and in the Colab badge above to your own `USER/REPO`.
>
> The install pulls `jax` + `diffrax`; `sstrax` is a git package (not on PyPI).
> The pinned `diffrax` silences a deprecation warning it triggers.

In [ ]:
!git clone -q https://github.com/chreissel/sbi-tutorial-iaifi26.git
%cd sbi-tutorial-iaifi26
!pip install -q -r requirements.txt
# sstrax (the GD1 stream simulator) is a git package, not on PyPI; falcon's Ray
# workers import our streams_model.py which imports it, so it must be installed.
!pip install -q "diffrax==0.6.1" "git+https://github.com/undark-lab/sstrax.git"

In [ ]:
import warnings; warnings.filterwarnings("ignore")   # sstrax/diffrax are chatty
import time
import numpy as np
import matplotlib.pyplot as plt

import streams_model as sm        # the simulator, wrapped for falcon
import streams_plotting as sp     # matplotlib-only plotting helpers

SEED = 0
np.random.seed(SEED)

---
## 0 — A stellar stream, in one simulator call

`sstrax.simulate_stream` takes **16 parameters** (`sm.PRIOR_LIST`) — the
progenitor's present-day position and velocity, its disruption age and mass,
and eight tidal-stripping "micro" parameters — and integrates the disrupting
cluster forward, returning the phase-space coordinates of the stream stars in
the Milky-Way (`halo`) frame:

`stars.shape == (N_stars, 6)` = (x, y, z, vx, vy, vz).

`N_stars` is **not fixed** — older or heavier progenitors shed more stars (a few
hundred to ~1000 here). The **first** call spends ~13 s compiling the `jax`
simulator; every call after that is fast.

In [ ]:
params = sm.sstrax.Parameters()                 # the 16 parameters, at their defaults

t0 = time.time()
stars = np.asarray(sm.sstrax.simulate_stream(key=sm.jax.random.PRNGKey(0), params=params))
print(f"first call : {time.time()-t0:5.1f} s  (includes ~13 s JIT compile)")
t0 = time.time()
_ = np.asarray(sm.sstrax.simulate_stream(key=sm.jax.random.PRNGKey(1), params=params))
print(f"second call: {time.time()-t0:5.2f} s")
print("stars.shape:", stars.shape, "-> (N_stars, 6)")

**Two ways to look at one stream.** On the left, the stars in the Galactic
(`halo`) frame — the physical ribbon wrapping around the Galactic centre. On
the right, the same stars in **GD1 stream coordinates** `(phi1, phi2)`: a
rotated sky frame in which the stream lies flat along `phi1`. That rotation
(`sm.stars_to_gd1`) is a fixed coordinate change we do in plain numpy — see the
note in the next section on why that matters for speed.

In [ ]:
sp.plot_stream_orbit_and_sky(stars, title="one GD1 stream at the fiducial parameters")
plt.show()

---
## 1 — From a variable star list to a fixed-size image

A network cannot read a list whose length changes every simulation. So, exactly
as `albatross` does, we turn each stream into a **fixed-shape image**:

1. rotate to GD1 observables — `(dist, phi1, phi2, vrad, pm_phi1_cosphi2, pm_phi2)`;
2. add per-observable Gaussian measurement errors and drop a few stars (`add_noise`);
3. add a uniform foreground of Milky-Way field stars (`sample_background`);
4. bin into **three 2-D histograms** — `(phi1, phi2)`, the proper motions, and
   `(dist, vrad)` — stacked into a single `(3, nbins, nbins)` array (`bin_stream`).

That fixed `(3, 48, 48)` shape is the whole trick: no matter how many stars a
stream has, the data the network sees is the same size.

> **Why this is fast enough to do live.** `albatross` rotates to GD1 with two
> jitted `jax` vmaps. Because `N_stars` changes every call, those vmaps
> *re-compile every single simulation* — several seconds each. `sm.stars_to_gd1`
> reimplements the identical rotation in numpy (it is affine — a rotation plus
> unit conversions), reproducing `sstrax` to float precision at ~0.5 ms. That
> ~6× speedup is what makes a full training run feasible in a hackathon.
> The timing test is in `hackathon_solutions/` (solutions branch).

In [ ]:
rng = np.random.default_rng(SEED)
truth_full = [sm.TRUE_VALUES[k] for k in sm.TRUE_VALUES]         # all 16 at truth
image = sm.simulate_image(truth_full, infer_params=list(sm.TRUE_VALUES), rng=rng)
print("image shape:", image.shape, " counts per channel:", image.sum((1,2)).astype(int))
sp.plot_channels(image, title="the three data channels a stream maps to")
plt.show()

### ✏️ Exercise 1 — watch the parameters move the data *(worked warm-up)*

Before inferring anything, get a feel for what is *learnable*: does changing a
parameter visibly change the data? We do this one together — it is the pattern
every idea below builds on. Simulate the stream image at **two disruption ages**
(1000 vs 4000 Myr), holding everything else at truth, and compare. The older
stream has had longer to disrupt, so it is **longer along `phi1`** and its
kinematic channels shift too.

`sm.simulate_image(z, infer_params=names, rng=...)` takes a value vector `z`
paired with parameter `names`; here we vary just `"age"`.

In [ ]:
for age in [1000., 4000.]:
    img = sm.simulate_image([age], infer_params=["age"], rng=np.random.default_rng(1))
    sp.plot_channels(img, title=f"age = {age:.0f} Myr")
    plt.show()

The image moves clearly with `age` — so `age` is something the data can
constrain. Try the same with another parameter (mass `logmsat`, or a velocity
component `vxc`) to build a mental map of which parameters the stream "sees"
before you spend simulations inferring them.

---
## 2 — Your hackathon

Everything from here is yours. You now have a complete, fast forward model — a
map from progenitor parameters to a fixed `(3, 48, 48)` stream image
(`sm.StreamImage`) and a data embedding for it (`sm.StreamCNN`) — and from
**notebook 2** you already know how to turn a forward model into a `falcon`
posterior. Nothing below is pre-written on purpose.

**The goal:** infer the GD1 progenitor's parameters from its stream image, and
work out what the data can and cannot tell you.

### Start here — get a baseline

Reach a first posterior over the two parameters `sm.DEFAULT_INFER` =
`['age', 'logmsat']` (both visibly move the data — you just saw `age` do it in Exercise 1).
Mirroring notebook 2, you will need to:

1. **Save one observation** at the truth to disk (`np.save(...)`) — the target
   the config's `observed:` points at. The truth values live in `sm.TRUE_VALUES`.
2. **Write a `falcon` config** (`%%writefile config_streams.yml`): a
   `falcon.priors.Product` prior over the two parameters (ranges in
   `sm.PRIOR_RANGES`), a `falcon.estimators.Flow` whose `embedding` is
   `sm.StreamCNN` reading the image (`_input_: [x]`), and a data node
   `sm.StreamImage` with `parents: [z]` and your `observed:` file. The config
   from notebook 2 is the template — only the priors, the embedding target, and
   the observed path change.
3. **Train and sample:** `!falcon launch ...` then `!falcon sample posterior ...`.
4. **Look at the answer:** load the per-sample NPZs (one `(D,)` array under key
   `z`, as in notebook 2) and plot with
   `sp.plot_posterior(post, sm.DEFAULT_INFER, truth=...)`.

Budget the simulator honestly: `sstrax` is ~0.5–1 s/sim, so a few hundred sims
is a real cost — that trade-off between simulation budget and posterior width is
the whole game.

### Then push it — pick whatever interests you

- **Infer more parameters.** Grow the `Product` prior and the data node's
  `infer_params` to the progenitor's full 6-D phase space (`xc … vzc`), or all
  16. Which parameters does the stream actually constrain, and which stay at the
  prior width?
- **Marginalise the nuisances.** The eight tidal-stripping micro-parameters are
  uncertain physics you don't care about. Put them in a **second node with a
  prior and no `evidence:`** and falcon marginalises them for free — notebook 2's
  nuisance pattern. `sm.StreamImage` already accepts a `nuisance_params` list for
  exactly this.
- **Design the embedding.** `sm.StreamCNN` is deliberately small. Does a bigger
  CNN, different normalisation, or a different `out_features` buy a tighter
  posterior at the *same* simulation budget?
- **Check calibration.** Are the posteriors trustworthy? Run the SBC / coverage
  diagnostics from **notebook 1** on a handful of held-out streams.
- **Sequential zoom.** Use a first posterior to restrict the prior, then re-train
  where the answer actually is (notebook 1 §6).

The one rule: **compare on simulation budget** — the number of `sstrax` calls —
the same currency notebook 2 used. The repo
[README hackathon prompt](https://github.com/chreissel/sbi-tutorial-iaifi26#hackathon-prompt) is the
one-paragraph version of all this. Good luck.

_Your workspace — add as many cells as you need._